# Permutations

Adapted from Qualtran v0.7.0 commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import SVG, display
from examples.algorithms.permutation_catalog import permutation_example_graph, permutation_examples
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

examples = permutation_examples()

def show_signatures(group):
    display(SVG(field_signature_svg([(name, signature, {}) for name, signature, _, _ in group])))

def title(node):
    if isinstance(node, tuple):
        kind = node[0]
        if kind == "cycle":
            return f"Cycle {node[3]}"
        if kind in ("equals", "controlled_xor"):
            return f"{kind}({node[2]})"
        return str(kind)
    return str(node)

def show_calls(example):
    graph = permutation_example_graph(example)
    display(SVG(render_call_graph_svg(export_call_graph_dot(
        graph.root, graph.direct_calls, title=title,
    ))))
    print("Source-declared leaves", graph.leaf_counts)

## `Permutation`
Apply a permutation of [0, N - 1] on the basis states.

Given a permutation $P : [0, N - 1] \to [0, N - 1]$, this bloq applies the unitary:

$$
    U|x\rangle = |P(x)\rangle
$$

Decomposes a permutation into cycles and applies them in order.
See :meth:`from_dense_permutation` to construct this bloq from a permutation,
and :meth:`from_partial_permutation_map` to construct it from a mapping.

#### Parameters
 - `N`: the total size the permutation acts on.
 - `cycles`: a sequence of permutation cycles that form the permutation.
 - `bitsize`: number of bits to store the indices, defaults to $\ceil(\log_2(N))$. 

#### Registers
 - `x`: integer register storing a value in [0, ..., N - 1] 

#### References
 - [A simple quantum algorithm to efficiently prepare sparse states](https://arxiv.org/abs/2310.19309v1). Appendix B.


The dense and partial factories use the original cycle discovery order. Partial traversal can lose an edge in a chain that meets a previously visited node. Callers that require every partial edge must check the realized mapping. Unknown cycle contents describe resource graphs without emitting gates.

In [ ]:
permutation_group = examples[:5]

### Example Instances

In [ ]:
permutation_group[0]

In [ ]:
permutation_group[1]

In [ ]:
permutation_group[2]

In [ ]:
permutation_group[3]

In [ ]:
permutation_group[4]

#### Graphical Signature

In [ ]:
show_signatures(permutation_group)

### Call Graph

In [ ]:
show_calls(permutation_group[0])

## `PermutationCycle`
Apply a single permutation cycle on the basis states.

Given a permutation cycle $C = (v_0 v_2 \ldots v_{k - 1})$, applies the following unitary:

    $$
        U|v_i\rangle \mapsto |v_{(i + 1)\mod k}\rangle
    $$

for each $i \in [0, k)$, and

    $$
        U|x\rangle \mapsto |x\rangle
    $$

and for every $x \not\in C$.

#### Parameters
 - `N`: the total size the permutation acts on.
 - `cycle`: the permutation cycle to apply.
 - `bitsize`: number of bits to store the indices, defaults to $\ceil(\log_2(N))$. 

#### Registers
 - `x`: integer register storing a value in [0, ..., N - 1] 

#### References
 - [A simple quantum algorithm to efficiently prepare sparse states](https://arxiv.org/abs/2310.19309v1). Appendix B, Algorithm 7.


A concrete cycle uses one clean marker, recurring equality work, and reusable classical feedback slots. The final equality returns the marker to zero. Concrete cycles can retain a symbolic register bound for declarations.

In [ ]:
cycle_group = examples[5:]

### Example Instances

In [ ]:
cycle_group[0]

In [ ]:
cycle_group[1]

In [ ]:
cycle_group[2]

#### Graphical Signature

In [ ]:
show_signatures(cycle_group)

### Call Graph

In [ ]:
show_calls(cycle_group[0])

## Execute through append

Apply overlapping cycles in the declared order, then reverse cycle order and each cycle's direction. The example verifies the full complex trajectory with an entangled spectator and all work wires clean. Dedicated tests cover the measurement histories. Its emitted primitive counts and resource model are reported separately.

In [ ]:
from examples.algorithms.register_permutation import main

main()